# Template de prova — Métodos Estatísticos (MEACC)

Notebook pronto para a prova. **Cada célula tem as variáveis a trocar no topo** — mude só elas e rode.

Ele já vem com uma base de exemplo embutida, então roda inteiro de cara. Na prova, troque a célula "2. Carregar sua base" pelo seu arquivo e o resto continua funcionando.

**Roteiro:** rode a célula 1 (setup) → carregue sua base (célula 2) → rode o diagnóstico (célula 3) → pule direto para a seção do tipo de análise que a questão pede.

| Questão pede | Vá para a seção |
|---|---|
| Distribuição de variável categórica | 4 |
| Distribuição de variável numérica, média/mediana/atípicos | 5 |
| Evolução ao longo do tempo | 6 |
| Relação entre duas categóricas | 7 |
| Relação entre duas numéricas | 8 |
| Comparar grupos (ex: homens x mulheres) | 9 |

## 1. Setup (rode sempre primeiro)

In [ ]:
import pandas as pd
import altair as alt

# ESSENCIAL: sem isso, base com mais de 5000 linhas dá MaxRowsError
alt.data_transformers.disable_max_rows()

print('pandas', pd.__version__, '| altair', alt.__version__)

# No Colab, para acessar arquivos do Drive, descomente:
# from google.colab import drive
# drive.mount('/content/drive')
# PASTA = '/content/drive/MyDrive/Academico/Métodos Estatísticos/2026/Prova'
# import os; print(os.listdir(PASTA))   # confere o nome EXATO dos arquivos

## 2. Carregar sua base

Troque o bloco de exemplo pela leitura do seu arquivo. Variantes úteis estão comentadas — CSV brasileiro costuma precisar de `sep=';'` e `decimal=','`.

In [ ]:
# --- NA PROVA, USE UMA DESTAS E APAGUE O EXEMPLO ABAIXO ---
# df = pd.read_csv(f'{PASTA}/arquivo.csv')
# df = pd.read_csv(f'{PASTA}/arquivo.csv', sep=';')                    # CSV com ponto e vírgula
# df = pd.read_csv(f'{PASTA}/arquivo.csv', sep=';', decimal=',')       # + vírgula decimal
# df = pd.read_csv(f'{PASTA}/arquivo.csv', sep=';', encoding='latin-1')# + acento quebrado
# df = pd.read_excel(f'{PASTA}/arquivo.xls')                           # .xls precisa de: !pip install xlrd
# df = pd.read_excel(f'{PASTA}/arquivo.xlsx')

# --- BASE DE EXEMPLO (apague na prova) ---
import numpy as np
rng = np.random.default_rng(7)
n = 200
df = pd.DataFrame({
    'estado': rng.choice(['RJ', 'SP', 'MG', 'BA', 'RS'], n),
    'escolaridade': rng.choice(['Fundamental', 'Médio', 'Superior'], n, p=[.3, .5, .2]),
    'internet': rng.choice(['Sim', 'Não'], n, p=[.8, .2]),
    'renda': rng.lognormal(7, .6, n).round(2),          # assimétrica à direita, como renda real
    'anos_estudo': rng.normal(9, 3, n).round(1).clip(0, 20),
    'data': pd.date_range('2023-01-01', periods=n, freq='D'),
    'temperatura': (25 + 5*np.sin(np.arange(n)/30) + rng.normal(0, 1.5, n)).round(1),
})
df.head()

## 3. Diagnóstico da base (rode sempre, antes de analisar)

Responde de uma vez: o separador estava certo? quais são os nomes exatos das colunas? os números são números mesmo ou vieram como texto? tem dado faltando?

In [ ]:
print('formato (linhas, colunas):', df.shape)
print()
print('colunas:', list(df.columns))
print()
print('tipos (object/str = TEXTO; int64/float64 = NÚMERO):')
print(df.dtypes)
print()
print('valores ausentes por coluna:')
print(df.isna().sum())
print()
# se alguma coluna tiver espaço sobrando no nome, descomente:
# df.columns = df.columns.str.strip()
# se alguma coluna tiver PONTO no nome, renomeie (senão o gráfico sai vazio!):
# df = df.rename(columns={'P.a.p.': 'Pap'})
df.head()

## 4. Univariada — variável QUALITATIVA (categórica)

Tabela de frequência + gráfico de barras. Use setores só se forem poucas categorias e somarem 100%.

In [ ]:
COL = 'escolaridade'          # <-- troque aqui

print(df[COL].value_counts())
print()
print('em percentual:')
print((df[COL].value_counts(normalize=True) * 100).round(1))

In [ ]:
COL = 'escolaridade'          # <-- troque aqui

alt.Chart(df).mark_bar().encode(
    x=alt.X(f'{COL}:N', sort='-y', axis=alt.Axis(labelAngle=-45)),
    y='count()',
    tooltip=[COL, 'count()']
).properties(width=400, title=f'Distribuição de {COL}')

In [ ]:
COL = 'escolaridade'          # <-- troque aqui (gráfico de setores)

alt.Chart(df).mark_arc().encode(
    theta='count()',
    color=f'{COL}:N',
    tooltip=[COL, 'count()']
)

## 5. Univariada — variável QUANTITATIVA (numérica)

O pacote completo: histograma + medidas de resumo + regra 1,5×AIQ + boxplot.
Ao interpretar, siga sempre: **forma → centro → dispersão → atípicos**.

In [ ]:
COL = 'renda'                 # <-- troque aqui
PASSO = 200                   # <-- largura das classes do histograma

alt.Chart(df).mark_bar().encode(
    alt.X(f'{COL}:Q', bin=alt.Bin(step=PASSO)),
    y='count()'
).properties(width=500, title=f'Distribuição de {COL}')

In [ ]:
COL = 'renda'                 # <-- troque aqui

print(df[COL].describe())
print()
print('média  :', round(df[COL].mean(), 2))
print('mediana:', round(df[COL].median(), 2))
print('moda   :', list(df[COL].mode()))
print('desvio padrão:', round(df[COL].std(), 2))
print()
if df[COL].mean() > df[COL].median():
    print('=> média > mediana: assimétrica à DIREITA (cauda de valores altos)')
elif df[COL].mean() < df[COL].median():
    print('=> média < mediana: assimétrica à ESQUERDA')
else:
    print('=> média = mediana: simétrica')

In [ ]:
COL = 'renda'                 # <-- troque aqui  (regra 1,5 x AIQ)

q1 = df[COL].quantile(0.25)
q3 = df[COL].quantile(0.75)
aiq = q3 - q1
lim_inf = q1 - 1.5 * aiq
lim_sup = q3 + 1.5 * aiq

print(f'Q1 = {q1:.2f} | mediana = {df[COL].median():.2f} | Q3 = {q3:.2f}')
print(f'AIQ = Q3 - Q1 = {aiq:.2f}')
print(f'Intervalo sem atípicos: [{lim_inf:.2f} ; {lim_sup:.2f}]')
print()
atipicos = df[(df[COL] < lim_inf) | (df[COL] > lim_sup)]
print(f'{len(atipicos)} valor(es) atípico(s) encontrado(s)')
atipicos

In [ ]:
COL = 'renda'                 # <-- troque aqui

alt.Chart(df).mark_boxplot(extent=1.5).encode(   # extent=1.5 destaca os atípicos
    x=f'{COL}:Q'
).properties(width=500, height=100, title=f'Boxplot de {COL}')

## 6. Série temporal

Lembre: o `resample` mudou de nome entre versões — mês é `'ME'` (antigo `'M'`), ano é `'YE'` (antigo `'Y'`).

In [ ]:
COL_DATA = 'data'             # <-- troque aqui
COL_VALOR = 'temperatura'     # <-- troque aqui

df[COL_DATA] = pd.to_datetime(df[COL_DATA])

alt.Chart(df).mark_line().encode(
    x=f'{COL_DATA}:T',
    y=f'{COL_VALOR}:Q',
    tooltip=[COL_DATA, COL_VALOR]
).properties(width=700, height=300, title=f'{COL_VALOR} ao longo do tempo')

In [ ]:
COL_DATA = 'data'             # <-- troque aqui
COL_VALOR = 'temperatura'     # <-- troque aqui
PERIODO = 'ME'                # 'D' dia | 'W' semana | 'ME' mês | 'YE' ano

try:
    serie = df.set_index(COL_DATA).resample(PERIODO)[COL_VALOR].mean()
except ValueError:                                   # pandas antigo
    serie = df.set_index(COL_DATA).resample(PERIODO.rstrip('E'))[COL_VALOR].mean()

alt.Chart(serie.reset_index()).mark_line(point=True).encode(
    x=f'{COL_DATA}:T',
    y=f'{COL_VALOR}:Q',
    tooltip=[COL_DATA, COL_VALOR]
).properties(width=700, height=300, title=f'Média de {COL_VALOR} por período')

## 7. Bivariada — duas QUALITATIVAS (tabela de dupla entrada)

Defina qual é a **explicativa**: é nela que você condiciona (dentro de cada categoria dela, como se distribui a outra).

In [ ]:
EXPLICATIVA = 'escolaridade'  # <-- troque aqui
RESPOSTA = 'internet'         # <-- troque aqui

tabela = df.groupby([EXPLICATIVA, RESPOSTA]).size().unstack(1)
tabela.loc['Total', :] = tabela.sum(axis=0)
tabela.loc[:, 'Total'] = tabela.sum(axis=1)
tabela = tabela.fillna(0)
print('Tabela de dupla entrada (contagens):')
tabela

In [ ]:
EXPLICATIVA = 'escolaridade'  # <-- troque aqui
RESPOSTA = 'internet'         # <-- troque aqui

# distribuição condicional: dentro de cada categoria da EXPLICATIVA, % de cada RESPOSTA
longo = df.groupby([EXPLICATIVA, RESPOSTA]).size()
condicional = (longo / longo.groupby(level=0).transform('sum') * 100).round(1)
print('Distribuição condicional (%) — cada grupo da explicativa soma 100:')
condicional

In [ ]:
EXPLICATIVA = 'escolaridade'  # <-- troque aqui
RESPOSTA = 'internet'         # <-- troque aqui

base = df.groupby([EXPLICATIVA, RESPOSTA]).size().reset_index().rename(columns={0: 'contagem'})

# em percentual (stack='normalize') -> é o que permite COMPARAR grupos de tamanhos diferentes
alt.Chart(base).mark_bar().encode(
    x=alt.X('contagem:Q', stack='normalize', axis=alt.Axis(format='%', title='proporção')),
    y=f'{EXPLICATIVA}:N',
    color=f'{RESPOSTA}:N',
    tooltip=[EXPLICATIVA, RESPOSTA, 'contagem']
).properties(width=500, title=f'{RESPOSTA} por {EXPLICATIVA}')

## 8. Bivariada — duas QUANTITATIVAS (dispersão + Pearson)

Explicativa no eixo x, resposta no eixo y. **Sempre olhe o gráfico antes de confiar no coeficiente.**

In [ ]:
X = 'anos_estudo'             # <-- explicativa
Y = 'renda'                   # <-- resposta

alt.Chart(df).mark_circle(size=60, opacity=.6).encode(
    x=alt.X(f'{X}:Q', scale=alt.Scale(zero=False)),
    y=alt.Y(f'{Y}:Q', scale=alt.Scale(zero=False)),
    tooltip=[X, Y]
).properties(width=500, height=400, title=f'{Y} vs {X}')

In [ ]:
X = 'anos_estudo'             # <-- troque aqui
Y = 'renda'                   # <-- troque aqui

r = df[[X, Y]].corr(numeric_only=True).iloc[0, 1]
print(f'Coeficiente de Pearson: r = {r:.3f}')

direcao = 'positiva' if r > 0 else 'negativa'
forca = 'forte' if abs(r) >= .7 else ('moderada' if abs(r) >= .3 else 'fraca')
print(f'=> relação linear {direcao} de intensidade {forca}')
print()
print('LEMBRE: correlação NÃO é causalidade, e r só mede relação LINEAR.')

In [ ]:
# Testar a influência de um valor atípico: remova e recalcule
X = 'anos_estudo'             # <-- troque aqui
Y = 'renda'                   # <-- troque aqui

idx_atipico = df[Y].idxmax()          # ou escolha o índice que você viu no gráfico
print('observação removida:')
print(df.loc[idx_atipico])
print()
r_com = df[[X, Y]].corr(numeric_only=True).iloc[0, 1]
r_sem = df.drop(idx_atipico)[[X, Y]].corr(numeric_only=True).iloc[0, 1]
print(f'r com o atípico:  {r_com:.3f}')
print(f'r sem o atípico:  {r_sem:.3f}')
print(f'diferença: {abs(r_com - r_sem):.3f}')

## 9. Comparar grupos (boxplots lado a lado)

Dois casos: (a) uma coluna de valor + uma coluna de grupo; (b) duas colunas separadas (ex: renda_homem e renda_mulher).

In [ ]:
# CASO (a): uma coluna numérica + uma coluna de grupo
VALOR = 'renda'               # <-- troque aqui
GRUPO = 'escolaridade'        # <-- troque aqui

alt.Chart(df).mark_boxplot(extent=1.5, size=30).encode(
    x=f'{VALOR}:Q',
    y=f'{GRUPO}:N'
).properties(width=600, height=200, title=f'{VALOR} por {GRUPO}')

In [ ]:
# CASO (a) — as medidas de cada grupo, para citar números na interpretação
VALOR = 'renda'               # <-- troque aqui
GRUPO = 'escolaridade'        # <-- troque aqui

resumo = df.groupby(GRUPO)[VALOR].describe()[['count', 'mean', '25%', '50%', '75%']]
resumo['AIQ'] = resumo['75%'] - resumo['25%']
resumo.round(2)

In [ ]:
# CASO (b): duas COLUNAS separadas (ex: 'Desagregação HOMEM ...' e 'Desagregação MULHER ...')
COLUNAS = ['renda', 'anos_estudo']    # <-- troque pelas duas colunas a comparar
ID = 'estado'                          # <-- coluna identificadora

# 'Category'/'Value' propositalmente em inglês para não colidir com colunas suas
longo = df.melt(id_vars=[ID], value_vars=COLUNAS, var_name='Category', value_name='Value')

alt.Chart(longo).mark_boxplot(extent=1.5, size=40).encode(
    x='Value:Q',
    y='Category:N',
    tooltip=[ID]
).properties(width=600, height=200)

## 10. Corretor de código (quando o gráfico não sai)

Carrega o `10-corretor-de-codigo-altair.py` do repositório. Se estiver usando o notebook solto, clone o repo antes:
`!git clone https://github.com/BeAmara1/cola-meacc.git`

In [ ]:
import os
CAMINHO_CORRETOR = '10-corretor-de-codigo-altair.py'   # ajuste se estiver em outra pasta

if os.path.exists(CAMINHO_CORRETOR):
    exec(open(CAMINHO_CORRETOR, encoding='utf-8').read().split("if __name__")[0])
    print('corretor carregado — use: checar(seu_codigo_como_string, df=sua_base)')
else:
    print(f'não achei {CAMINHO_CORRETOR} — clone o repositório ou ajuste o caminho')

In [ ]:
# exemplo de uso do corretor (código com parêntese faltando)
if 'checar' in dir():
    checar('''
alt.Chart(df).mark_bar().encode(
    x="escolaridade:N",
    y="count()"
''', df=df)

---

## Lembretes finais (as coisas que mais custam ponto)

1. **Toda questão pede interpretação escrita.** Código e gráfico sozinhos não fecham a resposta.
2. Ao descrever distribuição: **forma → centro → dispersão → atípicos**.
3. Comparar grupos de tamanhos diferentes: use **percentual**, nunca contagem bruta.
4. Correlação **não** é causalidade — e Pearson só enxerga relação **linear**.
5. Cheque valores atípicos pela regra **1,5×AIQ** e diga o efeito deles.
6. Média é sensível a atípico; **mediana é resistente**.